In [ ]:
from moviepy.editor import VideoFileClip

def convert_mov_to_mp4(input_path, output_path):
    """
    Converts a MOV video file to MP4 format.

    Args:
        input_path (str): The path to the input MOV file.
        output_path (str): The desired path for the output MP4 file.
    """
    try:
        clip = VideoFileClip(input_path)
        clip.write_videofile(output_path, codec="libx264")
        print(f"Conversion successful: {input_path} -> {output_path}")
    except Exception as e:
        print(f"Error during conversion: {e}")

# Example usage:
input_mov_file = "my_video.mov"
output_mp4_file = "my_video.mp4"
convert_mov_to_mp4(input_mov_file, output_mp4_file)

### Imports and Parameters

In [38]:
from ultralytics import YOLO
import numpy as np
from PIL import Image, ImageDraw
import random, math
from datetime import datetime

In [83]:
target_fish = [0, 1, 2, 3, 4, 5] # 0: Charlotte-Perkins-Gilman, 1: Margaret-Atwood, 2: Margaret-Thatcher, 3: Mary-Wollstonecraft, 4: Maya-Angelou, 5: Virginia-Woolf
#source = "test_clip.mp4"
source =  "IMG_1816.mp4" #03, 16, 17
background_path = "fArt_background.png"

fish_colors = {
    0: [(255, 238, 79), (253, 255, 142), (233, 136, 62)], # yellow, more yellow, red
    1: [(226, 97, 47), (235, 131, 55), (235, 173, 48)], # orange, more orange, yellow
    2: [(99, 156, 237), (21, 36, 64), (194, 179, 79)], # blue, dark blue, yellow
    3: [(247, 126, 55), (151, 113, 99), (205, 187, 230)], # orange, greyish, magenta
    4: [(48, 56, 91), (232, 114, 30), (168, 182, 255)], # purple, orange, blue
    5: [(16, 26, 23), (151, 224, 182), (213, 99, 26)]  # blue, teal, orange
} 

# Load a pretrained YOLO11n model
model = YOLO("best.pt")

### Extract points from video

In [ ]:
### WITHOUT FRAME NUMBER
def get_points(model):
    results = model(source, stream=True)
    dims = 0

    paths = {fish_id: [] for fish_id in target_fish} # NEW THING: dictionary comprehension

    for i, result in enumerate(results):
        if i == 0:
            dims = result.orig_shape
        boxes = result.boxes

        if boxes is None or boxes.cls is None:
            print(f"Frame {i}: No detections")
            continue

        class_ids = boxes.cls.cpu().numpy().astype(int)
        confidences = boxes.conf.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()

        for guppy_id in target_fish:
            mask = class_ids == guppy_id # NEW THING: "boolean masking or fancy indexing in NumPy"
            guppy_boxes = xyxy[mask]
            guppy_confs = confidences[mask]

            if len(guppy_boxes) > 0:
                max_index = guppy_confs.argmax()
                best_box = guppy_boxes[max_index]
                x_center = (best_box[0] + best_box[2]) / 2
                y_center = (best_box[1] + best_box[3]) / 2

                paths[guppy_id].append((x_center, y_center))
                print(f"Frame {i}, Guppy {guppy_id}: Center = ({x_center:.1f}, {y_center:.1f})")
            else:
                print(f"Frame {i}, Guppy {guppy_id}: Not detected")
    return paths, dims
        

In [40]:
def get_points(model, source, target_fish):
    results = model(source, stream=True)
    dims = 0
    paths = {fish_id: [] for fish_id in target_fish}

    for i, result in enumerate(results):  # i = frame index
        if i == 0:
            dims = result.orig_shape
        boxes = result.boxes

        if boxes is None or boxes.cls is None:
            print(f"Frame {i}: No detections")
            continue

        class_ids = boxes.cls.cpu().numpy().astype(int)
        confidences = boxes.conf.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()

        for guppy_id in target_fish:
            # Boolean mask to isolate boxes for this class ID
            mask = class_ids == guppy_id
            guppy_boxes = xyxy[mask]
            guppy_confs = confidences[mask]

            if len(guppy_boxes) > 0:
                # Choose the box with the highest confidence
                max_index = guppy_confs.argmax()
                best_box = guppy_boxes[max_index]
                x_center = (best_box[0] + best_box[2]) / 2
                y_center = (best_box[1] + best_box[3]) / 2

                # Append frame number and coordinates
                paths[guppy_id].append((i, x_center, y_center))
                print(f"Frame {i}, Guppy {guppy_id}: Center = ({x_center:.1f}, {y_center:.1f})")
            else:
                print(f"Frame {i}, Guppy {guppy_id}: Not detected")

    return paths, dims


In [84]:
paths, dims = get_points(model, source, target_fish)


video 1/1 (frame 1/380) c:\Users\coolu\Documents\fArt\code\IMG_1816.mp4: 512x896 2 Charlotte-Perkins-Gilmans, 1 Margaret-Atwood, 5 Margaret-Thatchers, 2 Mary-Wollstonecrafts, 2 Maya-Angelous, 2 Virginia-Woolfs, 21.6ms
Frame 0, Guppy 0: Center = (1752.1, 443.3)
Frame 0, Guppy 1: Center = (994.9, 497.8)
Frame 0, Guppy 2: Center = (179.0, 239.5)
Frame 0, Guppy 3: Center = (1552.9, 452.8)
Frame 0, Guppy 4: Center = (273.4, 197.9)
Frame 0, Guppy 5: Center = (495.2, 327.4)
video 1/1 (frame 2/380) c:\Users\coolu\Documents\fArt\code\IMG_1816.mp4: 512x896 1 Charlotte-Perkins-Gilman, 1 Margaret-Atwood, 2 Margaret-Thatchers, 3 Mary-Wollstonecrafts, 2 Maya-Angelous, 1 Virginia-Woolf, 19.5ms
Frame 1, Guppy 0: Center = (1750.3, 443.3)
Frame 1, Guppy 1: Center = (1002.5, 498.2)
Frame 1, Guppy 2: Center = (1650.5, 510.3)
Frame 1, Guppy 3: Center = (1550.9, 450.7)
Frame 1, Guppy 4: Center = (275.2, 242.4)
Frame 1, Guppy 5: Center = (498.1, 326.9)
video 1/1 (frame 3/380) c:\Users\coolu\Documents\fArt\c

In [ ]:
# old
def clean_and_fill_paths(paths, max_gap=10, max_jump=80):
    """
    Clean guppy paths by removing outlier jumps and interpolating short gaps.

    Args:
        paths (dict): {guppy_id: [(frame, x, y), ...]}
        max_gap (int): maximum consecutive missing frames to interpolate
        max_jump (float): maximum allowed distance (pixels) between consecutive detections
    Returns:
        dict: cleaned and interpolated paths
    """
    cleaned_paths = {}

    for guppy_id, points in paths.items():
        if len(points) < 2:
            cleaned_paths[guppy_id] = points
            continue

        # Sort points by frame index (safety)
        points = sorted(points, key=lambda p: p[0])
        new_points = [points[0]]

        for (f1, x1, y1), (f2, x2, y2) in zip(points[:-1], points[1:]):
            frame_gap = f2 - f1
            dist = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

            # --- Outlier filtering ---
            if dist > max_jump:
                print(f"Guppy {guppy_id}: skipping frame {f2} (jump {dist:.1f}px)")
                # Skip this point entirely
                continue

            # --- Interpolation for small gaps ---
            if 1 < frame_gap <= max_gap:
                for j in range(1, frame_gap):
                    alpha = j / frame_gap
                    x_interp = x1 + alpha * (x2 - x1)
                    y_interp = y1 + alpha * (y2 - y1)
                    f_interp = f1 + j
                    new_points.append((f_interp, x_interp, y_interp))
                    print(f"Guppy {guppy_id}: interpolated frame {f_interp}")

            elif frame_gap > max_gap:
                print(f"Guppy {guppy_id}: long gap ({frame_gap} frames) — skipping interpolation")

            # Keep the next actual detection
            new_points.append((f2, x2, y2))

        cleaned_paths[guppy_id] = sorted(new_points, key=lambda p: p[0])

    return cleaned_paths

In [ ]:
# implement something to check if it's the right guppy?
def clean_and_fill_paths(fish_tracks, max_jump_per_frame=20, max_gap_frames=50):
    """
    Clean and fill guppy tracking paths, handling YOLO misdetections and occlusions.

    Args:
        fish_tracks (dict): {fish_id: [(frame, x, y), ...]}
        max_jump_per_frame (float): max movement (pixels/frame) before considering it a new segment.
        max_gap_frames (int): max frame gap to interpolate through (beyond this, assume guppy left view).

    Returns:
        dict: cleaned and filled tracks.
    """
    cleaned_tracks = {}

    for fish_id, points in fish_tracks.items():
        points = sorted(points, key=lambda p: p[0])

        if not points:
            continue

        filled = [points[0]]

        for i in range(1, len(points)):
            f1, x1, y1 = filled[-1]
            f2, x2, y2 = points[i]

            frame_gap = f2 - f1
            dist = math.hypot(x2 - x1, y2 - y1)

            # Compute per-frame distance if frames missing
            per_frame_dist = dist / frame_gap if frame_gap > 0 else 0

            # Step 1: Check if jump is reasonable
            if per_frame_dist > max_jump_per_frame:
                # Too big — probably a wrong detection (skip it)
                continue

            filled.append((f2, x2, y2))

            # Step 2: Interpolate missing frames if gap is small enough
            if 1 < frame_gap <= max_gap_frames:
                for j in range(1, frame_gap):
                    t = j / frame_gap
                    fi = f1 + j
                    xi = x1 + (x2 - x1) * t
                    yi = y1 + (y2 - y1) * t
                    filled.append((fi, xi, yi))

        # Sort again (in case interpolation added midpoints)
        cleaned_tracks[fish_id] = sorted(filled, key=lambda p: p[0])

    return cleaned_tracks


In [85]:
#cleaned_paths = clean_and_fill_paths(paths, max_gap=200, max_jump=150)
cleaned_paths = clean_and_fill_paths(paths, max_jump_per_frame=30, max_gap_frames=50)

### Create Brush

In [44]:
def pick_weighted_color(colors):
    if len(colors) != 3:
        raise ValueError("Expected exactly 3 colors.")
    weights = [0.60, 0.25, 0.15]
    return random.choices(colors, weights=weights, k=1)[0]

In [16]:
### NO SIZE VARIATION
def draw_brushstroke(base_size, fishes, n=10, r=10, alpha_range=(80, 180),
                     size_range=(3, 15), shape_types=("rectangle",), scale=4):
    """
    base_size: size of original video frame
    points: list of (x, y)
    fishes: list of fish ids
    n: shapes per point
    r: max jitter radius
    alpha_range: (min, max) opacity
    size_range: (min, max) shape size
    shape_types: tuple of shape names to randomly pick from
    scale: scale factor when scaling to increase resolution
    """
    
    bg = Image.open(background_path).convert("RGBA")
    big_size = (base_size[0] * scale, base_size[1] * scale)
    big_img = Image.new("RGBA", big_size, (0, 0, 0, 0))  # fully transparent
    draw = ImageDraw.Draw(big_img, "RGBA")
    
    for fish in fishes:
        path = paths[fish]
        n_points = len(path)
        for i, (x, y) in enumerate(path):
            # Compute opacity scaling with progression
            progress = i / max(n_points - 1, 1)
            alpha_min, alpha_max = alpha_range
            alpha_base = int(alpha_min + (alpha_max - alpha_min) * progress)
            for _ in range(n):
                # position jitter
                angle = random.uniform(0, 2 * math.pi)
                dist = random.uniform(0, r)
                jitter_x = (x + dist * math.cos(angle)) * scale
                jitter_y = (y + dist * math.sin(angle)) * scale

                # random size and rotation
                w = random.uniform(*size_range) * scale
                h = random.uniform(*size_range) * scale
                rot = math.radians(random.uniform(0, 360))

                # color + opacity
                rgb = pick_weighted_color(fish_colors[fish])
                alpha = random.randint(int(alpha_base * 0.8), int(alpha_base * 1.0)) # small randomness
                alpha = min(255, max(0, alpha))  # clamp to [0, 255]
                rgba = (*rgb, alpha)

                shape = random.choice(shape_types)

                if shape == "rectangle":
                    corners = [(-w/2, -h/2), (w/2, -h/2), (w/2, h/2), (-w/2, h/2)]
                    rotated = []
                    for (cx, cy) in corners:
                        rx = cx * math.cos(rot) - cy * math.sin(rot)
                        ry = cx * math.sin(rot) + cy * math.cos(rot)
                        rotated.append((jitter_x + rx, jitter_y + ry))
                    draw.polygon(rotated, fill=rgba)

                elif shape == "circle":
                    # bounding box for ellipse
                    draw.ellipse(
                        (jitter_x - w/2, jitter_y - h/2,
                        jitter_x + w/2, jitter_y + h/2),
                        fill=rgba
                    )

                elif shape == "triangle":
                    corners = [(0, -h/2), (w/2, h/2), (-w/2, h/2)]
                    rotated = []
                    for (cx, cy) in corners:
                        rx = cx * math.cos(rot) - cy * math.sin(rot)
                        ry = cx * math.sin(rot) + cy * math.cos(rot)
                        rotated.append((jitter_x + rx, jitter_y + ry))
                    draw.polygon(rotated, fill=rgba)
                
                elif shape == "blob":
                    num_points = 10
                    radius = w / 2
                    points = []
                    for i in range(num_points):
                        angle = 2 * math.pi * i / num_points
                        # radius varies randomly for a "wavy" contour
                        r_i = radius * (0.7 + 0.3 * random.random())
                        x_blob = r_i * math.cos(angle)
                        y_blob = r_i * math.sin(angle)
                        points.append((x_blob, y_blob))

                    # rotate + translate
                    rotated = []
                    for (cx, cy) in points:
                        rx = cx * math.cos(rot) - cy * math.sin(rot)
                        ry = cx * math.sin(rot) + cy * math.cos(rot)
                        rotated.append((jitter_x + rx, jitter_y + ry))
                    draw.polygon(rotated, fill=rgba)
    
    brush_layer = big_img.resize(base_size, Image.Resampling.LANCZOS)

    result = bg.copy()
    result.alpha_composite(brush_layer)
    
    return result

In [58]:
### SIZE VARIATION
def draw_brushstroke(paths, base_size, fishes, n=10, r=10, alpha_range=(80, 180),
                     size_range=(3, 15), shape_types=("rectangle",), scale=4):
    """
    base_size: size of original video frame
    points: list of (x, y)
    fishes: list of fish ids
    n: shapes per point
    r: max jitter radius
    alpha_range: (min, max) opacity
    size_range: (min, max) shape size
    shape_types: tuple of shape names to randomly pick from
    scale: scale factor when scaling to increase resolution
    """
    
    bg = Image.open(background_path).convert("RGBA")
    big_size = (base_size[0] * scale, base_size[1] * scale)
    big_img = Image.new("RGBA", big_size, (0, 0, 0, 0))  # fully transparent
    draw = ImageDraw.Draw(big_img, "RGBA")
    
    for fish in fishes:
        path = paths[fish]
        n_points = len(path)
        for i, (_, x, y) in enumerate(path):  # ignore frame index
            # Compute opacity scaling with progression
            progress = i / max(n_points - 1, 1)
            alpha_min, alpha_max = alpha_range
            alpha_base = int(alpha_min + (alpha_max - alpha_min) * progress)
            # size ramp
            size_min, size_max = size_range
            # e.g., smaller at start, larger at end
            size_base = size_min + (size_max - size_min) * progress
            
            for _ in range(n):
                # position jitter
                angle = random.uniform(0, 2 * math.pi)
                dist = random.uniform(0, r)
                jitter_x = (x + dist * math.cos(angle)) * scale
                jitter_y = (y + dist * math.sin(angle)) * scale

                # random size and rotation
                w = random.uniform(size_base * 0.8, size_base * 1.2) * scale
                h = random.uniform(size_base * 0.8, size_base * 1.2) * scale
                rot = math.radians(random.uniform(0, 360))

                # color + opacity
                rgb = pick_weighted_color(fish_colors[fish])
                alpha = random.randint(int(alpha_base * 0.8), int(alpha_base * 1.0)) # small randomness
                alpha = min(255, max(0, alpha))  # clamp to [0, 255]
                rgba = (*rgb, alpha)

                shape = random.choice(shape_types)

                if shape == "rectangle":
                    corners = [(-w/2, -h/2), (w/2, -h/2), (w/2, h/2), (-w/2, h/2)]
                    rotated = []
                    for (cx, cy) in corners:
                        rx = cx * math.cos(rot) - cy * math.sin(rot)
                        ry = cx * math.sin(rot) + cy * math.cos(rot)
                        rotated.append((jitter_x + rx, jitter_y + ry))
                    draw.polygon(rotated, fill=rgba)

                elif shape == "circle":
                    # bounding box for ellipse
                    draw.ellipse(
                        (jitter_x - w/2, jitter_y - h/2,
                        jitter_x + w/2, jitter_y + h/2),
                        fill=rgba
                    )

                elif shape == "triangle":
                    corners = [(0, -h/2), (w/2, h/2), (-w/2, h/2)]
                    rotated = []
                    for (cx, cy) in corners:
                        rx = cx * math.cos(rot) - cy * math.sin(rot)
                        ry = cx * math.sin(rot) + cy * math.cos(rot)
                        rotated.append((jitter_x + rx, jitter_y + ry))
                    draw.polygon(rotated, fill=rgba)
                
                elif shape == "blob":
                    num_points = 10
                    radius = w / 2
                    points = []
                    for i in range(num_points):
                        angle = 2 * math.pi * i / num_points
                        # radius varies randomly for a "wavy" contour
                        r_i = radius * (0.7 + 0.3 * random.random())
                        x_blob = r_i * math.cos(angle)
                        y_blob = r_i * math.sin(angle)
                        points.append((x_blob, y_blob))

                    # rotate + translate
                    rotated = []
                    for (cx, cy) in points:
                        rx = cx * math.cos(rot) - cy * math.sin(rot)
                        ry = cx * math.sin(rot) + cy * math.cos(rot)
                        rotated.append((jitter_x + rx, jitter_y + ry))
                    draw.polygon(rotated, fill=rgba)
    
    brush_layer = big_img.resize(base_size, Image.Resampling.LANCZOS)

    result = bg.copy()
    result.alpha_composite(brush_layer)
    
    return result

### Create Fish Art

In [88]:
#artists = [0, 1, 2, 3, 4, 5]
artists = [2]

img = draw_brushstroke(
    cleaned_paths,
    (dims[1], dims[0]),
    fishes=artists,
    n=15, r=12,
    alpha_range=(155, 255),
    size_range=(6, 20),
    shape_types=("rectangle",)  # mix of shapes
)

# Convert to RGB before saving (to blend correctly with background)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
img.convert("RGB").save(f"fArts/fArt_{timestamp}.png")

### Everything together
Works better to run seperately, easier to modify brush settings without re-running object detection every time

In [ ]:
from ultralytics import YOLO
import numpy as np
from PIL import Image, ImageDraw
import random, math

target_fish = [0, 1] # 0: Charlotte-Perkins-Gilman, 1: Margaret-Atwood, 2: Margaret-Thatcher, 3: Mary-Wollstonecraft, 4: Maya-Angelou, 5: Virginia-Woolf
source = "test_clip.mp4"

fish_colors = {
    0: [(255, 238, 79), (253, 255, 142), (233, 136, 62)], # yellow, more yellow, red
    1: [(226, 97, 47), (235, 131, 55), (235, 173, 48)], # orange, more orange, yellow
    2: [(99, 156, 237), (21, 36, 64), (194, 179, 79)], # blue, dark blue, yellow
    3: [(247, 126, 55), (151, 113, 99), (205, 187, 230)], # orange, greyish, magenta
    4: [(48, 56, 91), (232, 114, 30), (168, 182, 255)], # purple, orange, blue
    5: [(16, 26, 23), (151, 224, 182), (213, 99, 26)]  # blue, teal, orange
} 

# Load a pretrained YOLO11n model
model = YOLO("best.pt")

paths, dims = get_points(model)

img = draw_brushstroke(
    (dims[1], dims[0]),
    fishes=target_fish,
    n=15, r=12,
    alpha_range=(100, 200),
    size_range=(6, 20),
    shape_types=("rectangle",)  # mix of shapes
)

# Convert to RGB before saving (to blend correctly with white background)
img.convert("RGB").save("first_fArt.png")